In [3]:
from pathlib import Path
from dotenv import load_dotenv
import os

# Project directory
PROJECT_DIR = Path(
    r"C:\Users\RAYANSTAR\Desktop\nikan rosh project(automation)\clinic_project"
)

# Load environment variables
ENV_FILE = PROJECT_DIR / ".env"

load_dotenv(dotenv_path=ENV_FILE, override=True)

print("Environment file exists:", ENV_FILE.exists())

for variable in [
    "DB_HOST",
    "DB_PORT",
    "DB_NAME",
    "DB_USER",
    "DB_PASSWORD"
]:
    print(f"{variable}: {'Loaded' if os.getenv(variable) else 'Missing'}")

Environment file exists: True
DB_HOST: Loaded
DB_PORT: Loaded
DB_NAME: Loaded
DB_USER: Loaded
DB_PASSWORD: Loaded


In [4]:
from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

DATABASE_URL = URL.create(
    drivername="mysql+pymysql",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
    query={"charset": "utf8mb4"}
)

engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    echo=False
)

with engine.connect() as connection:
    result = connection.execute(text("SELECT DATABASE(), 1"))
    row = result.fetchone()

    print("Connection successful!")
    print("Database:", row[0])
    print("Test query:", row[1])

Connection successful!
Database: clinic_system
Test query: 1


In [5]:
from sqlalchemy import inspect

inspector = inspect(engine)

# Get all existing tables
tables = inspector.get_table_names()

print("Tables in database:")
print("=" * 50)

for table_name in tables:

    print(f"\nTABLE: {table_name}")
    print("-" * 50)

    # Columns
    columns = inspector.get_columns(table_name)

    print("COLUMNS:")

    for column in columns:
        print({
            "name": column["name"],
            "type": str(column["type"]),
            "nullable": column["nullable"],
            "default": column["default"],
            "autoincrement": column.get("autoincrement")
        })

    # Primary key
    pk = inspector.get_pk_constraint(table_name)
    print("\nPRIMARY KEY:")
    print(pk)

    # Foreign keys
    fks = inspector.get_foreign_keys(table_name)
    print("\nFOREIGN KEYS:")

    for fk in fks:
        print({
            "columns": fk["constrained_columns"],
            "references_table": fk["referred_table"],
            "references_columns": fk["referred_columns"]
        })

    # Unique constraints
    unique_constraints = inspector.get_unique_constraints(table_name)
    print("\nUNIQUE CONSTRAINTS:")

    for constraint in unique_constraints:
        print(constraint)

Tables in database:

TABLE: appointments
--------------------------------------------------
COLUMNS:
{'name': 'appointment_id', 'type': 'BIGINT', 'nullable': False, 'default': None, 'autoincrement': True}
{'name': 'patient_id', 'type': 'BIGINT', 'nullable': False, 'default': None, 'autoincrement': False}
{'name': 'created_by', 'type': 'BIGINT', 'nullable': False, 'default': None, 'autoincrement': False}
{'name': 'appointment_date', 'type': 'DATETIME', 'nullable': False, 'default': 'CURRENT_TIMESTAMP', 'autoincrement': None}
{'name': 'status', 'type': 'ENUM', 'nullable': False, 'default': "'created'", 'autoincrement': None}
{'name': 'created_at', 'type': 'DATETIME', 'nullable': False, 'default': 'CURRENT_TIMESTAMP', 'autoincrement': None}

PRIMARY KEY:
{'constrained_columns': ['appointment_id'], 'name': None}

FOREIGN KEYS:
{'columns': ['patient_id'], 'references_table': 'patients', 'references_columns': ['patient_id']}
{'columns': ['created_by'], 'references_table': 'users', 'reference

In [6]:
from sqlalchemy import text

query = text("""
    SELECT TABLE_NAME, COLUMN_NAME, COLUMN_TYPE, COLUMN_DEFAULT
    FROM INFORMATION_SCHEMA.COLUMNS
    WHERE TABLE_SCHEMA = :database_name
      AND DATA_TYPE = 'enum'
    ORDER BY TABLE_NAME, ORDINAL_POSITION
""")

with engine.connect() as connection:
    rows = connection.execute(query, {"database_name": "clinic_system"}).fetchall()

for row in rows:
    print(row.TABLE_NAME, "|", row.COLUMN_NAME, "|", row.COLUMN_TYPE, "| default:", row.COLUMN_DEFAULT)

appointments | status | enum('created','in_progress','completed','cancelled') | default: created
audio_files | processing_status | enum('uploaded','transcribing','transcribed','failed') | default: uploaded
documents | document_type | enum('medical_report','prescription','other') | default: medical_report
documents | print_status | enum('generated','printed','failed') | default: generated
medical_records | status | enum('draft','pending_review','approved') | default: draft
users | role | enum('secretary','doctor','admin') | default: None


In [7]:
print("engine defined:", "engine" in globals())
print("Base defined:", "Base" in globals())
print("SessionLocal defined:", "SessionLocal" in globals())

engine defined: True
Base defined: False
SessionLocal defined: False


In [8]:
from sqlalchemy.orm import sessionmaker, DeclarativeBase

class Base(DeclarativeBase):
    pass

SessionLocal = sessionmaker(bind=engine, autoflush=False, autocommit=False)

def get_db():
    """FastAPI-style dependency: one session per request."""
    db = SessionLocal()
    try:
        yield db
    finally:
        db.close()

print("Base defined:", "Base" in globals())
print("SessionLocal defined:", "SessionLocal" in globals())

Base defined: True
SessionLocal defined: True


In [9]:
import enum
from datetime import datetime, date
from typing import Optional, List

from sqlalchemy import (
    BigInteger, CHAR, String, Text, DateTime, Date, Boolean,
    ForeignKey, Enum as SAEnum, JSON, func,
)
from sqlalchemy.orm import Mapped, mapped_column, relationship


# ---------- Enums (values must match MySQL exactly) ----------
class UserRole(str, enum.Enum):
    secretary = "secretary"
    doctor = "doctor"
    admin = "admin"

class AppointmentStatus(str, enum.Enum):
    created = "created"
    in_progress = "in_progress"
    completed = "completed"
    cancelled = "cancelled"

class AudioProcessingStatus(str, enum.Enum):
    uploaded = "uploaded"
    transcribing = "transcribing"
    transcribed = "transcribed"
    failed = "failed"

class RecordStatus(str, enum.Enum):
    draft = "draft"
    pending_review = "pending_review"
    approved = "approved"

class DocumentType(str, enum.Enum):
    medical_report = "medical_report"
    prescription = "prescription"
    other = "other"

class PrintStatus(str, enum.Enum):
    generated = "generated"
    printed = "printed"
    failed = "failed"


def pg_enum(enum_cls):
    # store the enum *values* (not names) as MySQL expects
    return SAEnum(enum_cls, values_callable=lambda e: [m.value for m in e], native_enum=True)


# ---------- Models ----------
class Patient(Base):
    __tablename__ = "patients"

    patient_id: Mapped[int] = mapped_column(BigInteger, primary_key=True, autoincrement=True)
    national_code: Mapped[str] = mapped_column(CHAR(10), unique=True, nullable=False)
    first_name: Mapped[str] = mapped_column(String(100), nullable=False)
    last_name: Mapped[str] = mapped_column(String(100), nullable=False)
    phone_number: Mapped[Optional[str]] = mapped_column(String(20))
    birth_date: Mapped[Optional[date]] = mapped_column(Date)
    created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now(), nullable=False)

    appointments: Mapped[List["Appointment"]] = relationship(back_populates="patient")


class User(Base):
    __tablename__ = "users"

    user_id: Mapped[int] = mapped_column(BigInteger, primary_key=True, autoincrement=True)
    full_name: Mapped[str] = mapped_column(String(150), nullable=False)
    username: Mapped[str] = mapped_column(String(50), unique=True, nullable=False)
    password_hash: Mapped[str] = mapped_column(String(255), nullable=False)
    role: Mapped[UserRole] = mapped_column(pg_enum(UserRole), nullable=False)
    is_active: Mapped[bool] = mapped_column(Boolean, server_default="1", nullable=False)
    created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now(), nullable=False)

    appointments_created: Mapped[List["Appointment"]] = relationship(back_populates="creator")
    approved_records: Mapped[List["MedicalRecord"]] = relationship(back_populates="approver")


class Appointment(Base):
    __tablename__ = "appointments"

    appointment_id: Mapped[int] = mapped_column(BigInteger, primary_key=True, autoincrement=True)
    patient_id: Mapped[int] = mapped_column(BigInteger, ForeignKey("patients.patient_id"), nullable=False)
    created_by: Mapped[int] = mapped_column(BigInteger, ForeignKey("users.user_id"), nullable=False)
    appointment_date: Mapped[datetime] = mapped_column(DateTime, server_default=func.now(), nullable=False)
    status: Mapped[AppointmentStatus] = mapped_column(
        pg_enum(AppointmentStatus), server_default="created", nullable=False
    )
    created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now(), nullable=False)

    patient: Mapped["Patient"] = relationship(back_populates="appointments")
    creator: Mapped["User"] = relationship(back_populates="appointments_created")
    audio_files: Mapped[List["AudioFile"]] = relationship(back_populates="appointment")
    medical_record: Mapped[Optional["MedicalRecord"]] = relationship(
        back_populates="appointment", uselist=False
    )


class AudioFile(Base):
    __tablename__ = "audio_files"

    audio_id: Mapped[int] = mapped_column(BigInteger, primary_key=True, autoincrement=True)
    appointment_id: Mapped[int] = mapped_column(BigInteger, ForeignKey("appointments.appointment_id"), nullable=False)
    file_path: Mapped[str] = mapped_column(String(500), nullable=False)
    transcription: Mapped[Optional[str]] = mapped_column(Text().with_variant(Text(4294967295), "mysql"))
    processing_status: Mapped[AudioProcessingStatus] = mapped_column(
        pg_enum(AudioProcessingStatus), server_default="uploaded", nullable=False
    )
    created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now(), nullable=False)

    appointment: Mapped["Appointment"] = relationship(back_populates="audio_files")


class MedicalRecord(Base):
    __tablename__ = "medical_records"

    record_id: Mapped[int] = mapped_column(BigInteger, primary_key=True, autoincrement=True)
    appointment_id: Mapped[int] = mapped_column(
        BigInteger, ForeignKey("appointments.appointment_id"), unique=True, nullable=False
    )
    approved_by: Mapped[Optional[int]] = mapped_column(BigInteger, ForeignKey("users.user_id"))
    diagnosis: Mapped[Optional[str]] = mapped_column(Text)
    instructions: Mapped[Optional[str]] = mapped_column(Text)
    extraction_json: Mapped[Optional[dict]] = mapped_column(JSON)
    status: Mapped[RecordStatus] = mapped_column(pg_enum(RecordStatus), server_default="draft", nullable=False)
    approved_at: Mapped[Optional[datetime]] = mapped_column(DateTime)
    created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now(), nullable=False)
    updated_at: Mapped[datetime] = mapped_column(
        DateTime, server_default=func.now(), onupdate=func.now(), nullable=False
    )

    appointment: Mapped["Appointment"] = relationship(back_populates="medical_record")
    approver: Mapped[Optional["User"]] = relationship(back_populates="approved_records")
    medications: Mapped[List["Medication"]] = relationship(back_populates="record")
    documents: Mapped[List["Document"]] = relationship(back_populates="record")


class Medication(Base):
    __tablename__ = "medications"

    medication_id: Mapped[int] = mapped_column(BigInteger, primary_key=True, autoincrement=True)
    record_id: Mapped[int] = mapped_column(BigInteger, ForeignKey("medical_records.record_id"), nullable=False)
    drug_name: Mapped[str] = mapped_column(String(200), nullable=False)
    dosage: Mapped[Optional[str]] = mapped_column(String(100))
    frequency: Mapped[Optional[str]] = mapped_column(String(100))
    duration: Mapped[Optional[str]] = mapped_column(String(100))
    instructions: Mapped[Optional[str]] = mapped_column(Text)

    record: Mapped["MedicalRecord"] = relationship(back_populates="medications")


class Document(Base):
    __tablename__ = "documents"

    document_id: Mapped[int] = mapped_column(BigInteger, primary_key=True, autoincrement=True)
    record_id: Mapped[int] = mapped_column(BigInteger, ForeignKey("medical_records.record_id"), nullable=False)
    file_path: Mapped[str] = mapped_column(String(500), nullable=False)
    document_type: Mapped[DocumentType] = mapped_column(
        pg_enum(DocumentType), server_default="medical_report", nullable=False
    )
    print_status: Mapped[PrintStatus] = mapped_column(
        pg_enum(PrintStatus), server_default="generated", nullable=False
    )
    created_at: Mapped[datetime] = mapped_column(DateTime, server_default=func.now(), nullable=False)

    record: Mapped["MedicalRecord"] = relationship(back_populates="documents")

In [10]:
from sqlalchemy import select

with SessionLocal() as db:
    print("patients:", db.scalar(select(func.count()).select_from(Patient)))
    print("users:", db.scalar(select(func.count()).select_from(User)))
    print("appointments:", db.scalar(select(func.count()).select_from(Appointment)))
    print("records:", db.scalar(select(func.count()).select_from(MedicalRecord)))

patients: 0
users: 0
appointments: 0
records: 0


In [12]:
import bcrypt

def hash_password(plain_password: str) -> str:
    return bcrypt.hashpw(plain_password.encode("utf-8"), bcrypt.gensalt()).decode("utf-8")

with SessionLocal() as db:
    secretary = User(
        full_name="Test Secretary",
        username="secretary1",
        password_hash=hash_password("Secretary@12345"),
        role=UserRole.secretary,
    )
    doctor = User(
        full_name="Test Doctor",
        username="doctor1",
        password_hash=hash_password("Doctor@12345"),
        role=UserRole.doctor,
    )
    db.add_all([secretary, doctor])
    db.commit()
    print("Users created:", secretary.user_id, doctor.user_id)

Users created: 1 2


In [13]:
with SessionLocal() as db:
    for u in db.scalars(select(User)).all():
        print(u.user_id, u.username, u.role.value, u.is_active)

1 secretary1 secretary True
2 doctor1 doctor True


In [14]:
import re

def is_valid_national_code(code: str) -> bool:
    """Validate Iranian national code (10 digits + checksum)."""
    if not re.fullmatch(r"\d{10}", code):
        return False
    if len(set(code)) == 1:  # e.g. 1111111111
        return False
    digits = [int(c) for c in code]
    s = sum(digits[i] * (10 - i) for i in range(9))
    r = s % 11
    check = r if r < 2 else 11 - r
    return check == digits[9]

# Quick test
print(is_valid_national_code("0012345678"))  # checksum test
print(is_valid_national_code("1111111111"))  # False
print(is_valid_national_code("123"))         # False

False
False
False


In [15]:
from datetime import date
from typing import Optional
from sqlalchemy import select
from sqlalchemy.exc import IntegrityError


class PatientError(Exception):
    pass


def get_patient_by_national_code(db, national_code: str) -> Optional[Patient]:
    return db.scalar(select(Patient).where(Patient.national_code == national_code))


def create_patient(db, national_code: str, first_name: str, last_name: str,
                   phone_number: Optional[str] = None,
                   birth_date: Optional[date] = None) -> Patient:
    national_code = national_code.strip()
    if not is_valid_national_code(national_code):
        raise PatientError("Invalid national code.")
    if get_patient_by_national_code(db, national_code):
        raise PatientError("A patient with this national code already exists.")
    if not first_name.strip() or not last_name.strip():
        raise PatientError("First name and last name are required.")

    patient = Patient(
        national_code=national_code,
        first_name=first_name.strip(),
        last_name=last_name.strip(),
        phone_number=phone_number,
        birth_date=birth_date,
    )
    db.add(patient)
    try:
        db.commit()
    except IntegrityError:
        db.rollback()
        raise PatientError("Could not save patient (duplicate or invalid data).")
    db.refresh(patient)
    return patient


def _clean_required(value: Optional[str], field_name: str) -> str:
    if value is None or not value.strip():
        raise PatientError(f"{field_name} cannot be empty.")
    return value.strip()


def update_patient(db, patient_id: int, **fields) -> Patient:
    allowed = {"first_name", "last_name", "phone_number", "birth_date"}
    patient = db.get(Patient, patient_id)
    if not patient:
        raise PatientError("Patient not found.")

    # Validate everything first, then apply
    cleaned = {}
    for key, value in fields.items():
        if key not in allowed:
            raise PatientError(f"Field '{key}' cannot be updated.")
        if key in {"first_name", "last_name"}:
            value = _clean_required(value, key)
        cleaned[key] = value

    for key, value in cleaned.items():
        setattr(patient, key, value)

    try:
        db.commit()
    except Exception:
        db.rollback()
        raise PatientError("Could not update patient.")
    db.refresh(patient)
    return patient


def list_patients(db, limit: int = 50, offset: int = 0):
    return db.scalars(select(Patient).order_by(Patient.patient_id).limit(limit).offset(offset)).all()

In [16]:
with SessionLocal() as db:
    # 1) create
    p = create_patient(db, "0499370899", "علی", "رضایی", phone_number="09121234567")
    print("Created:", p.patient_id, p.first_name, p.last_name)

    # 2) search
    found = get_patient_by_national_code(db, "0499370899")
    print("Found:", found.first_name, found.last_name)

    # 3) update
    update_patient(db, p.patient_id, phone_number="09350000000")
    print("Updated phone:", get_patient_by_national_code(db, "0499370899").phone_number)

    # 4) duplicate should fail
    try:
        create_patient(db, "0499370899", "علی", "رضایی")
    except PatientError as e:
        print("Expected error:", e)

    # 5) invalid code should fail
    try:
        create_patient(db, "1234567890", "تست", "تست")
    except PatientError as e:
        print("Expected error:", e)

    print("Total patients:", len(list_patients(db)))

Created: 1 علی رضایی
Found: علی رضایی
Updated phone: 09350000000
Expected error: A patient with this national code already exists.
Expected error: Invalid national code.
Total patients: 1


In [17]:
with SessionLocal() as db:
    p = get_patient_by_national_code(db, "0499370899")
    try:
        update_patient(db, p.patient_id, first_name="   ")
    except PatientError as e:
        print("Expected error:", e)
    print("Name unchanged:", get_patient_by_national_code(db, "0499370899").first_name)

Expected error: first_name cannot be empty.
Name unchanged: علی
